# 第008讲 函数代数与数学表达

从同一个受限模型 f(x)=2x+1，检查公式含义、定义域与数学结论。

先完成 lab.md 的纸笔步骤，再使用 Restart Kernel 和 Run All。直接先修只有003；本实验中的辅助函数已经准备好，不要求先学程序封装。数据完全合成，运行离线，不训练模型。

## 1 读入资料并核对位置

Fraction 接收分子与分母，可以保持小有理数精确。下面显示的是文件是否存在，不显示机器内部路径。

In [1]:
from fractions import Fraction
import math
import experiment as exp

print("合成数据存在：", (exp.BASE / "data" / "observations.csv").is_file())
rows = exp.read_observations()
xs = [r["x"] for r in rows]
ys = [r["y"] for r in rows]
print("样本数：", len(rows))
print("输入：", [str(x) for x in xs])
print("观测：", [str(y) for y in ys])
assert len(rows) == 5

合成数据存在： True
样本数： 5
输入： ['0', '1', '2', '3', '4']
观测： ['1', '2', '6', '6', '9']


## 2 四种表示的交叉核对

文字是先乘2再加1，公式为 f(x)=2x+1，对应表在CSV里。下面让程序逐项计算，再增加未列出的输入3/2。注意括号变更会改变规则。

In [2]:
for row in rows:
    actual = exp.f(row["x"])
    assert actual == row["expected"]
    print(row["sample_id"], "输入", str(row["x"]), "预测", str(actual))
extra_x = Fraction(3, 2)
print("新增输入 3/2 的正确预测：", exp.f(extra_x))
print("错误括号 2*(x+1) 的结果：", 2 * (extra_x + 1))
assert exp.f(extra_x) == 4

S01 输入 0 预测 1
S02 输入 1 预测 3
S03 输入 2 预测 5
S04 输入 3 预测 7
S05 输入 4 预测 9
新增输入 3/2 的正确预测： 4
错误括号 2*(x+1) 的结果： 5


## 3 定义域与复合

比较 g(f(x)) 和 f(g(x))，其中 g 是平方。预期拒绝由辅助检查捕获，正常整本运行不会因为这些预期案例中断。

In [3]:
print("闭区间端点：", exp.f(Fraction(0)), exp.f(Fraction(4)))
print("x=1，先f后g：", exp.g(exp.f(Fraction(1))))
print("x=1，先g后f：", exp.f(exp.g(Fraction(1))))
print("x=3，先f后g：", exp.g(exp.f(Fraction(3))))
print(exp.expect_rejection("x=3 反向复合被正确拒绝", lambda: exp.f(exp.g(Fraction(3)))))
print(exp.expect_rejection("x=5 被受限函数正确拒绝", lambda: exp.f(Fraction(5))))
print("约分在 1/2 的值：", exp.cancelled_original(Fraction(1, 2)))
print(exp.expect_rejection("约分不能补回 x=1", lambda: exp.cancelled_original(Fraction(1))))

闭区间端点： 1 9
x=1，先f后g： 9
x=1，先g后f： 3
x=3，先f后g： 49
x=3 反向复合被正确拒绝
x=5 被受限函数正确拒绝
约分在 1/2 的值： 3/2
约分不能补回 x=1


## 4 对数定义与错误反例

真数正，底数正且不等于1。负的对数值可以合法，负的真数在本实数约定中不合法。

In [4]:
print("log2(8)：", exp.log_real(8, 2))
print("log2(1/8)：", exp.log_real(Fraction(1, 8), 2))
print("以1/2为底的8的对数：", exp.log_real(8, Fraction(1, 2)))
print("错误加法规则的两侧：", exp.log_real(1+1, 2), exp.log_real(1,2)+exp.log_real(1,2))
print("log2((-2)^2)：", exp.log_real((-2)**2, 2))
print(exp.expect_rejection("log2(-2) 被正确拒绝", lambda: exp.log_real(-2, 2)))
print(exp.expect_rejection("底数1被正确拒绝", lambda: exp.log_real(7, 1)))

log2(8)： 3.0
log2(1/8)： -3.0
以1/2为底的8的对数： -3.0
错误加法规则的两侧： 1.0 0.0
log2((-2)^2)： 2.0
log2(-2) 被正确拒绝
底数1被正确拒绝


## 5 合法恒等式的有限浮点核验

360项来自明确列出的有限网格。两种容差都是1e-12；这不是所有实数上的误差保证，也不是一般恒等式的证明。数学理由见讲义的指数推导。

In [5]:
log_checks = exp.log_identity_checks()
print("核验项数：", log_checks["count"])
print("最大绝对差：", log_checks["max_absolute_difference"])
print("相对容差：", log_checks["relative_tolerance"])
print("绝对容差：", log_checks["absolute_tolerance"])
assert log_checks["count"] == 360
assert log_checks["all_passed"]

核验项数： 360
最大绝对差： 2.6645352591003757e-15
相对容差： 1e-12
绝对容差： 1e-12


## 6 索引与精确误差

数学 i=1 对应 Python j=0。每条的输入、观测、预测必须始终配对。两个观测值都为6仍保留两个样本位置。

In [6]:
summary = exp.error_summary(xs, ys)
for j in range(len(xs)):
    print("数学 i =", j+1, "Python j =", j,
          "输入", str(xs[j]), "观测", str(ys[j]),
          "预测", str(summary["predictions"][j]),
          "差", str(summary["residuals"][j]))
print("平方差之和：", summary["squared_sum"])
print("差之和再平方：", summary["residual_sum"]**2)
print("精确平均平方差：", summary["mse"])
assert summary["mse"] == Fraction(3, 5)
print("空求和与空乘积：", sum([]), math.prod([]))
print(exp.expect_rejection("空平均被正确拒绝", lambda: exp.error_summary([], [])))

数学 i = 1 Python j = 0 输入 0 观测 1 预测 1 差 0
数学 i = 2 Python j = 1 输入 1 观测 2 预测 3 差 1
数学 i = 3 Python j = 2 输入 2 观测 6 预测 5 差 -1
数学 i = 4 Python j = 3 输入 3 观测 6 预测 7 差 1
数学 i = 5 Python j = 4 输入 4 观测 9 预测 9 差 0
平方差之和： 3
差之和再平方： 1
精确平均平方差： 3/5
空求和与空乘积： 0 1
空平均被正确拒绝


## 7 量词顺序

每个x各自选择b，与先固定同一个b再应对所有x，是两条不同命题。这里穷尽有限集合，运算为精确整数。

In [7]:
quantifiers = exp.quantifier_demo()
print("每个 x 存在 b：", quantifiers["forall_x_exists_b"])
print("存在共同 b 适合所有 x：", quantifiers["exists_b_forall_x"])
print("每行的见证：", quantifiers["witnesses"])
print("共同 b：", quantifiers["shared_b"])
assert quantifiers["forall_x_exists_b"] is True
assert quantifiers["exists_b_forall_x"] is False

每个 x 存在 b： True
存在共同 b 适合所有 x： False
每行的见证： [{'x': 0, 'valid_b': [0]}, {'x': 1, 'valid_b': [-2]}, {'x': 2, 'valid_b': [-4]}]
共同 b： []


## 8 五点一致不能覆盖整个区间

另一函数加入x(x-1)(x-2)(x-3)(x-4)。先看原五点全通过，再看合法输入1/2的精确差。这是全区间相同主张的反例。

In [8]:
for x in xs:
    assert exp.alternative(x) == exp.f(x)
print("五个原始测试点都一致：", True)
x = Fraction(1,2)
print("f(1/2)：", exp.f(x))
print("p(1/2)：", exp.alternative(x))
difference = exp.alternative(x) - exp.f(x)
print("精确差：", difference)
assert difference == Fraction(105,32)

五个原始测试点都一致： True
f(1/2)： 2
p(1/2)： 169/32
精确差： 105/32


## 9 奇数和与精确输入

检查1至100只核验有限实现。全部正整数的结论需要你在纸上完成起点、假设与任意推进步骤。随后比较两种Fraction输入，说明精确容器不能修复先前已发生的近似。

In [9]:
for n in range(1, 101):
    odd_sum = sum(2*i-1 for i in range(1,n+1))
    assert odd_sum == n*n
print("n 从 1 到 100 全通过；最后一项：", odd_sum)
print("字符串 0.1：", Fraction("0.1"))
print("浮点 0.1：", Fraction(0.1))
assert Fraction("0.1") != Fraction(0.1)
print("一般证明请写在纸上，有限通过不是同一个结论。")

n 从 1 到 100 全通过；最后一项： 10000
字符串 0.1： 1/10
浮点 0.1： 3602879701896397/36028797018963968
一般证明请写在纸上，有限通过不是同一个结论。


## 10 完整核验与生成报告

这里检查所有边界，并重新运行核心流程。最后比较逐步结果，确认没有依赖另一份隐藏计算。

In [10]:
report = exp.run_experiment(write_outputs=True)
exp.print_report(report)
assert report["mse_exact"] == str(summary["mse"])
assert report["quantifiers"] == quantifiers
assert report["log_identities"] == log_checks
assert report["finite_grid_counterexample"]["difference"] == str(difference)
assert len(report["domain_checks"]) == 22
print("脚本与逐步核对一致：", True)
print("报告文件存在：", (exp.BASE / "outputs" / "experiment_report.json").is_file())

合法输入范围：[0,4]；四种表达新增输入 3/2，预测 = 4
预测： 1, 3, 5, 7, 9
带方向差： 0, 1, -1, 1, 0
平方差之和： 3 ；和的平方： 1
精确平均平方差： 3/5
复合顺序在 x=1： {'g_after_f': 9, 'f_after_g': 3}
每个 x 有一个 b： True
存在同一个 b 适合全部 x： False
五点都相同，在 x=1/2 的精确差： 105/32
合法对数有限数值核验： 360 项通过
定义域与边界核验： 22 项通过
奇数和精确核验 n=1 至 100：通过；一般证明仍见讲义归纳步骤
脚本核验通过；不把有限测试当成普遍证明。
脚本与逐步核对一致： True
报告文件存在： True


## 你的解释

在副本中写出：一个定义域错误、一个合法数值反例、量词顺序的区别，以及奇数求和完整归纳证明。不要仅写通过。

执行边界：制作时在新的Python进程中使用真实ipykernel进程内内核，按顺序执行并保留输出。没有测试浏览器Jupyter UI、外进程socket传输或本机Anaconda安装。